# MMM budget allocation: structured recommendations, policies, and benchmarks

A reproducible **synthetic Marketing Mix Modeling** walkthrough for RecGuard. Each recommendation is an entire four-week allocation across four channels and two regions: **32 dependent spend lines**, a model snapshot, uncertainty estimates, and a currency. We rank complete plans; selecting channel rows independently could break the budget.

This is a policy-enforcement benchmark, **not a fitted causal MMM or evidence of financial performance**. Synthetic adstock and saturation curves generate modeled incremental revenue. The candidate generator is a finite search, not a global optimizer. No customer data, Gemini calls, credentials, or paid services are needed. If Gemini proposes plans later, treat its amounts and predicted returns as untrusted and rescore them on the backend.

Run all cells in order. Edit `Rules` and `EXTRA_POLICIES` to add policies. The final cells produce repeatable benchmark tables, failure examples, and CSV/JSON artifacts.

In [ ]:
%pip install -q -e ..

In [ ]:
import csv, hashlib, json, math, random, statistics, time
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from types import MappingProxyType
from recguard import Candidate, RecGuard, Request, load_pack

POLICY_DIR = Path("policies/mmm_budget")

SEED = 20261026
TENANT, PURPOSE = "acme-marketing", "mmm-budget"
CHANNELS = ("search", "social", "video", "affiliate")
REGIONS = ("north", "south")
WEEKS = (1, 2, 3, 4)
MODEL_VERSION = "synthetic-mmm-v1"
DATA_VERSION = "synthetic-weekly-data-v1"

def show_table(rows):
    if not rows:
        print("(no rows)")
        return
    headers = list(rows[0])
    print(" | ".join(headers))
    for row in rows:
        print(" | ".join(str(row[h]) for h in headers))

## 1. Typed, immutable allocation plans and policy inputs

All amounts are **integer INR**. Model and data versions, currency, request context, and allocation contents are bound to a SHA-256 digest. Backend registries are frozen after construction. Only the digest and plan ID enter RecGuard; a delivery adapter verifies the full plan against the approved candidate before releasing it.

The example uses a fixed total budget, weekly pacing, regional envelopes, a channel concentration cap, a minimum search commitment, a maximum change from the current plan, and model uncertainty limits. These are **illustrative business rules**, not general investment advice.

In [ ]:
@dataclass(frozen=True)
class Spend:
    week: int
    region: str
    channel: str
    amount: int

@dataclass(frozen=True)
class Plan:
    plan_id: str
    context_id: str
    lines: tuple[Spend, ...]
    currency: str = "INR"
    model_version: str = MODEL_VERSION
    data_version: str = DATA_VERSION

@dataclass(frozen=True)
class Rules:
    total_budget: int
    weekly_budget: int
    region_budget: int
    max_channel_share: float
    min_search_share: float
    max_line_change: float
    max_relative_uncertainty: float
    min_lower_roas: float

# Limits are policy, so they live in YAML (policies/mmm_budget/governance.yaml).
RULES = Rules(**load_pack(POLICY_DIR / "governance.yaml").constants["limits"])
BASE_LINES = tuple(Spend(w, r, c, 12_500) for w in WEEKS for r in REGIONS for c in CHANNELS)

def digest(plan):
    payload = json.dumps(asdict(plan), sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode()).hexdigest()

## 2. Synthetic response model: carryover, saturation, and uncertainty

Channel-specific geometric adstock carries spend across weeks. A saturating response curve maps adstock into incremental revenue. Revenue subtracts a zero-future-spend counterfactual with the **same historical carryover**; otherwise existing advertising would be incorrectly credited to the new plan.

One hundred seeded parameter draws represent synthetic model uncertainty. We report mean revenue, a 10th percentile lower bound, and relative 10–90% interval width. Draws are shared between plans for fair comparisons. They are invented, **not a posterior estimated from data**. The objective covers four weeks only; it omits revenue after the horizon, interactions, seasonality, organic demand, and causal identification.

In [ ]:
DECAY = dict(zip(CHANNELS, (0.25, 0.45, 0.65, 0.15)))
CAPACITY = dict(zip(CHANNELS, (65_000, 55_000, 48_000, 44_000)))
HALF_SAT = dict(zip(CHANNELS, (24_000, 28_000, 34_000, 22_000)))
rng = random.Random(SEED)
DRAWS = tuple({c: rng.lognormvariate(-0.5 * 0.15**2, 0.15) for c in CHANNELS} for _ in range(100))

def forecast(plan):
    amounts = {(x.week, x.region, x.channel): x.amount for x in plan.lines}
    channel_revenue = {c: 0.0 for c in CHANNELS}
    for r in REGIONS:
        for c in CHANNELS:
            carry = counterfactual = 8_000.0
            for w in WEEKS:
                carry = DECAY[c] * carry + amounts.get((w, r, c), 0)
                counterfactual *= DECAY[c]
                response = lambda a: CAPACITY[c] * a / (HALF_SAT[c] + a)
                channel_revenue[c] += response(carry) - response(counterfactual)
    draws = sorted(sum(channel_revenue[c] * d[c] for c in CHANNELS) for d in DRAWS)
    mean = statistics.mean(draws)
    lower, upper = draws[9], draws[89]
    spend = sum(x.amount for x in plan.lines)
    return {"mean_revenue": mean, "lower_revenue": lower, "upper_revenue": upper,
            "relative_uncertainty": (upper - lower) / mean if mean > 0 else float("inf"),
            "lower_roas": lower / spend if spend > 0 else 0.0}

## 3. Candidate plans and adversarial fixtures

Generate balanced reallocations that preserve weekly and regional totals, plus deliberately unsafe plans: overspend, pacing drift, regional drift, concentration, contract underdelivery, abrupt changes, unsupported currency, and stale model/data snapshots. A fake plan ID and spoofed score test the adapter itself.

Synthetic negative examples receive a high submitted score to imitate an optimizer or LLM overstating their merit. The full adapter recomputes scores from trusted forecasts; native policies alone cannot verify an optimizer's numbers.

In [ ]:
plans, forecasts, contexts = {}, {}, {}

def store(plan):
    assert plan.plan_id not in plans
    plans[plan.plan_id] = plan
    forecasts[plan.plan_id] = forecast(plan)
    return plan.plan_id

def alter(plan, name, fn, **kwargs):
    return replace(plan, plan_id=name, lines=tuple(fn(x) for x in plan.lines), **kwargs)

cases = []
for i in range(48):
    ctx = f"context-{i:03d}"
    contexts[ctx] = RULES
    baseline = Plan(f"baseline-{i:03d}", ctx, BASE_LINES)
    fallback_id = store(baseline)
    generated = []
    local = random.Random(SEED + i)
    for j in range(6):
        lines = list(BASE_LINES)
        for w in WEEKS:
            for r in REGIONS:
                delta = local.randrange(-6_000, 6_001, 500)
                lines = [replace(x, amount=x.amount + (delta if x.channel == "search" else -delta if x.channel == "video" else 0))
                         if (x.week, x.region) == (w, r) else x for x in lines]
        generated.append(store(Plan(f"proposal-{i:03d}-{j}", ctx, tuple(lines))))
    attack_type = i % 8
    if attack_type == 0:
        bad = alter(baseline, f"attack-{i}", lambda x: replace(x, amount=x.amount * 2))
    elif attack_type == 1:
        bad = alter(baseline, f"attack-{i}", lambda x: replace(x, amount=x.amount + (6_000 if x.week == 1 else -6_000 if x.week == 4 else 0)))
    elif attack_type == 2:
        bad = alter(baseline, f"attack-{i}", lambda x: replace(x, amount=x.amount + (6_000 if x.region == "north" else -6_000)))
    elif attack_type == 3:
        bad = alter(baseline, f"attack-{i}", lambda x: replace(x, amount=35_000 if x.channel == "social" else 5_000))
    elif attack_type == 4:
        bad = alter(baseline, f"attack-{i}", lambda x: replace(x, amount=2_500 if x.channel == "search" else 22_500 if x.channel == "video" else x.amount))
    elif attack_type == 5:
        # Keeps all portfolio totals unchanged; violates only the per-line change limit.
        bad = alter(baseline, f"attack-{i}", lambda x: replace(x, amount=x.amount + (11_000 if x.week == 1 else -11_000 if x.week == 2 else 0) * (1 if x.channel == "search" else -1 if x.channel == "video" else 0)))
    elif attack_type == 6:
        bad = replace(baseline, plan_id=f"attack-{i}", currency="USD")
    else:
        bad = replace(baseline, plan_id=f"attack-{i}", model_version="retired-mmm", data_version="stale-data")
    attack_id = store(bad)
    cases.append({"context": ctx, "kind": ("overspend", "pacing", "region", "concentration", "commitment", "churn", "currency", "stale")[attack_type],
                  "proposals": tuple([attack_id] + generated), "fallback": fallback_id})

# Freeze trusted snapshots before any policy is registered or decision issued.
PLANS = MappingProxyType(plans.copy())
FORECASTS = MappingProxyType({k: MappingProxyType(v.copy()) for k, v in forecasts.items()})
CONTEXTS = MappingProxyType(contexts.copy())
del plans, forecasts, contexts

def canonical_candidate(plan_id):
    plan = PLANS[plan_id]
    return Candidate(item_id=plan_id, tenant_id=TENANT, categories=["budget-plan"],
                     evidence_ids=[digest(plan)], score=FORECASTS[plan_id]["mean_revenue"] / 1_000_000)

def request_for(case, label):
    candidates = [canonical_candidate(pid) for pid in case["proposals"]]
    candidates[0] = candidates[0].model_copy(update={"score": 9.0})
    # Submitted unknown plans carry fabricated evidence and cannot bypass the adapter.
    candidates.append(Candidate(item_id="unknown-plan", tenant_id=TENANT, score=10.0,
                                categories=["budget-plan"], evidence_ids=["fabricated"]))
    return Request(tenant_id=TENANT, request_id=f"{case['context']}-{label}", subject_ref=case["context"],
                   purpose=PURPOSE, k=1, candidates=candidates,
                   fallback_candidates=[canonical_candidate(case["fallback"])])

sample = PLANS[cases[0]["proposals"][1]]
show_table([asdict(x) for x in sample.lines[:8]])
print("Plans:", len(PLANS), "| Cases:", len(cases), "| Forecast:", dict(FORECASTS[sample.plan_id]))

## 4. Policies you can add: YAML policy packs

Policies, limits and custom checks live in **YAML files** under `policies/mmm_budget/`. There are no Python check functions. Each check is an ordered list of `reject: REASON` / `if: <condition>` rules over the candidate (`item`), the `request`, and trusted data this notebook passes in by name (`PLANS`, `FORECASTS`, `CONTEXTS`, …). Conditions use a safe expression subset that is parsed, never `eval`-ed. Typos and disallowed syntax fail when the pack loads; run `recguard policy validate policies/mmm_budget/*.yaml` from a shell.

Native `Policy` fields handle category allowlists, evidence presence, result count and turning off exploration. Custom checks validate the actual plan. **Evidence presence alone does not validate a digest or model provenance.** The binding check ties each candidate to its trusted plan and request context: score, attributes and digest. Portfolio rules then inspect every spend line. Checks reject a **whole plan** and can't rebalance it. A separately generated fallback is evaluated by the same rules.

Each check's content hash is part of the policy fingerprint. Editing a YAML rule and reloading it changes the fingerprint and invalidates outstanding receipts, so there's no silent in-place mutation. Model snapshot IDs must come from a trusted registry in production.

| Pack file | Check | Enforced rule |
|---|---|---|
| `governance.yaml` | policy `mmm-governance` | Evidence, category, one complete plan, no exploration; the `limits` used by every rule |
| `binding.yaml` | `binding-v1` | Trusted score, complete grid, digest, context, approved snapshot |
| `budget.yaml` | `budget-v1` | Currency, total, weekly and regional envelopes |
| `operations.yaml` | `operations-v1` | Concentration, search commitment, maximum reallocation |
| `risk.yaml` | `risk-v1` | Uncertainty interval and lower-bound ROAS |
| `extensions/strict_roas.yaml` (optional) | `strict-roas-v1` | Example stricter lower-ROAS floor; add it to `EXTRA_PACKS` |

**To add a policy:** copy an extension file, change the rule, and list it in `EXTRA_PACKS` (or in a `CONFIGS` layer).


In [ ]:
# Trusted data the packs may read. Packs declare what they need in `context:`.
CTX = {"PLANS": PLANS, "FORECASTS": FORECASTS, "CONTEXTS": CONTEXTS, "WEEKS": WEEKS, "REGIONS": REGIONS,
       "CHANNELS": CHANNELS, "canonical_candidate": canonical_candidate}

# Each configuration is a stack of YAML packs; "full" may also take EXTRA_PACKS.
CONFIGS = {"native": ("governance.yaml",),
           "budget": ("governance.yaml", "binding.yaml", "budget.yaml"),
           "operations": ("governance.yaml", "binding.yaml", "budget.yaml", "operations.yaml"),
           "full": ("governance.yaml", "binding.yaml", "budget.yaml", "operations.yaml", "risk.yaml")}
EXTRA_PACKS = ()  # e.g. ("extensions/strict_roas.yaml",), then rerun the engine/benchmark cells

def engine_for(label, extra=()):
    engine = RecGuard(receipt_ttl=1800)
    for pack in CONFIGS[label] + tuple(extra):
        engine.load_policy_pack(POLICY_DIR / pack, context=CTX)
    return engine

for label, packs in CONFIGS.items():
    print(f"{label:10s}", [load_pack(POLICY_DIR / p).name for p in packs])

def benchmark_request(case, label):
    req = request_for(case, label)
    # Different ID, same allocation: valid binding but unsafe portfolio.
    req.candidates.append(canonical_candidate(case["proposals"][0]))
    return req

## 5. Independent feasibility oracle and benchmark definitions

The oracle is written separately from the RecGuard checks. It checks the **complete intended rule set**, even for partial policy configurations. It is a test oracle for these fixtures, not proof of complete real-world compliance.

We compare an unguarded highest-submitted-score baseline and four RecGuard configurations. Each sees the same proposals and fallback. Metrics include unsafe selection rate (per request), feasibility among served plans, abstention, safe-plan coverage, fallback use, false rejection of feasible submitted candidates, latency p50/p95, and modeled revenue regret versus the best feasible plan **in this finite pool**. Regret is reported only for feasible selections; abstentions are reported separately and never counted as successful safety repairs.

`full` includes your `EXTRA_POLICIES`; a stricter extension can intentionally reduce coverage. The oracle remains the base business contract, so it exposes that tradeoff. Candidate utility is synthetic modeled revenue, not realized revenue or causal ROI.

Fallback is considered only when primary candidates cannot fill the result. It is not jointly ranked with valid primary plans. Consequently, even a safe decision can have positive regret against the pool oracle when the fallback has higher predicted revenue. The isolated admission matrix below removes ranking effects and shows which unsafe canonical plans each configuration actually rejects.


In [ ]:
def oracle_ok(plan, rules):
    if plan is None or plan.currency != "INR": return False
    if plan.model_version != MODEL_VERSION or plan.data_version != DATA_VERSION: return False
    coordinates = [(row.week, row.region, row.channel) for row in plan.lines]
    required = [(w, r, c) for w in WEEKS for r in REGIONS for c in CHANNELS]
    if sorted(coordinates) != sorted(required): return False
    if any(type(row.amount) is not int or row.amount < 0 for row in plan.lines): return False
    if sum(row.amount for row in plan.lines) != rules.total_budget: return False
    for w in WEEKS:
        if sum(row.amount for row in plan.lines if row.week == w) != rules.weekly_budget: return False
    for r in REGIONS:
        if sum(row.amount for row in plan.lines if row.region == r) != rules.region_budget: return False
    channel_spend = [sum(row.amount for row in plan.lines if row.channel == c) for c in CHANNELS]
    if any(value / rules.total_budget > rules.max_channel_share for value in channel_spend): return False
    if channel_spend[0] / rules.total_budget < rules.min_search_share: return False
    if any(abs(row.amount / 12_500 - 1) > rules.max_line_change + 1e-12 for row in plan.lines): return False
    f = FORECASTS[plan.plan_id]
    return f["relative_uncertainty"] <= rules.max_relative_uncertainty and f["lower_roas"] >= rules.min_lower_roas

def oracle_candidate(candidate, req):
    p = PLANS.get(candidate.item_id)
    return bool(p and p.context_id == req.subject_ref and req.purpose == PURPOSE
                and candidate.model_dump() == canonical_candidate(p.plan_id).model_dump()
                and oracle_ok(p, CONTEXTS[req.subject_ref]))

engines = {label: engine_for(label, EXTRA_PACKS if label == "full" else ()) for label in CONFIGS}
records, decisions, reason_counts = [], {}, Counter()
for case in cases:
    for label in ("unguarded", *CONFIGS):
        req = benchmark_request(case, label)
        pool = req.candidates + req.fallback_candidates
        feasible = [c for c in pool if oracle_candidate(c, req)]
        best = max((FORECASTS[c.item_id]["mean_revenue"] for c in feasible), default=None)
        started = time.perf_counter()
        if label == "unguarded":
            selected = sorted(req.candidates, key=lambda c: (-c.score, c.item_id))[:1]
            rejected = []
        else:
            dec = engines[label].evaluate(req)
            decisions[(case["context"], label)] = (req, dec)
            selected, rejected = dec.results, dec.rejections
            reason_counts.update((label, rejection.reason) for rejection in rejected)
        elapsed = (time.perf_counter() - started) * 1000
        safe = bool(selected) and all(oracle_candidate(c, req) for c in selected)
        rejected_ids = {r.item_id for r in rejected if r.source == "candidates"}
        # Duplicate IDs can represent spoofed and canonical copies. Count only
        # rejection IDs having no safe selected copy; fixtures use one canonical copy.
        selected_ids = {c.item_id for c in selected}
        false_rejects = sum(oracle_candidate(c, req) and c.item_id in rejected_ids and c.item_id not in selected_ids
                            for c in req.candidates)
        records.append({"configuration": label, "context": case["context"], "fixture": case["kind"],
                        "served": bool(selected), "safe": safe, "unsafe": bool(selected) and not safe,
                        "feasible_available": bool(feasible), "fallback": bool(selected) and selected[0].item_id == case["fallback"],
                        "false_rejections": false_rejects,
                        "feasible_submitted": sum(oracle_candidate(c, req) for c in req.candidates),
                        "latency_ms": elapsed,
                        "regret_inr": max(0.0, best - FORECASTS[selected[0].item_id]["mean_revenue"]) if safe and best is not None else None})

def percentile(values, q):
    ordered = sorted(values)
    return ordered[max(0, math.ceil(q * len(ordered)) - 1)]

summary = []
for label in ("unguarded", *CONFIGS):
    rows = [r for r in records if r["configuration"] == label]
    count, served = len(rows), sum(r["served"] for r in rows)
    regrets = [r["regret_inr"] for r in rows if r["regret_inr"] is not None]
    summary.append({"configuration": label, "requests": count,
                    "unsafe_pct": round(100 * sum(r["unsafe"] for r in rows) / count, 2),
                    "served_feasible_pct": round(100 * sum(r["safe"] for r in rows) / served, 2) if served else None,
                    "abstain_pct": round(100 * (count - served) / count, 2),
                    "safe_coverage_pct": round(100 * sum(r["safe"] for r in rows) / sum(r["feasible_available"] for r in rows), 2),
                    "fallback_pct": round(100 * sum(r["fallback"] for r in rows) / count, 2),
                    "false_rejection_pct": round(100 * sum(r["false_rejections"] for r in rows) / sum(r["feasible_submitted"] for r in rows), 2),
                    "p50_ms": round(statistics.median(r["latency_ms"] for r in rows), 3),
                    "p95_ms": round(percentile([r["latency_ms"] for r in rows], .95), 3),
                    "mean_pool_regret_inr": round(statistics.mean(regrets), 2) if regrets else None})
show_table(summary)
print("\nFull-policy rejection reasons:")
show_table([{"reason": reason, "count": count} for (label, reason), count in reason_counts.items() if label == "full"])

## 6. Safety assertions, risk rejection, fallback, and infeasible requests

These checks exercise canonical unsafe plans independently of score spoofing, including failures hidden by first-rejection ordering. Risk limits are deliberately tightened on a new engine to test their enforcement. The original rule set permits the baseline; a stricter policy can reject **every** available plan and must abstain.

Timing includes in-memory database logging and policy checks on this machine. It excludes model fitting and proposal generation. Repeat with production-sized pools, held-out scenarios, and a persisted store for deployment benchmarks. Forty-eight deterministic synthetic cases do not establish statistical confidence on real workloads.

In [ ]:
full = engines["full"]
assert all(not r["unsafe"] for r in records if r["configuration"] == "full")
if not EXTRA_PACKS:
    assert all(r["safe"] for r in records if r["configuration"] == "full")
# Each of the eight business-rule violations must fail with canonical attributes.
for case in cases[:8]:
    c = canonical_candidate(case["proposals"][0])
    req = Request(tenant_id=TENANT, request_id=f"canonical-{case['kind']}", purpose=PURPOSE,
                  subject_ref=case["context"], k=1, candidates=[c])
    d = full.evaluate(req)
    assert d.status == "BLOCKED", (case["kind"], d)
    assert not oracle_candidate(c, req)
    if case["kind"] == "churn":
        assert any("ABRUPT_REALLOCATION" in r.reason for r in d.rejections)
    print(case["kind"], "→", [r.reason for r in d.rejections])

case = cases[0]
base = canonical_candidate(case["fallback"])
req = Request(tenant_id=TENANT, request_id="risk-strict", purpose=PURPOSE,
              subject_ref=case["context"], k=1, candidates=[base])
strict = engine_for("full", ("extensions/strict_roas.yaml",))
blocked = strict.evaluate(req)
assert blocked.status == "BLOCKED"
assert any("STRICT_ROAS_FLOOR" in r.reason for r in blocked.rejections)
# No feasible candidate available: fallback must also pass the exact same checks.
unsafe = canonical_candidate(case["proposals"][0])
no_safe = req.model_copy(update={"request_id": "no-feasible", "candidates": [unsafe], "fallback_candidates": [unsafe]})
assert full.evaluate(no_safe).status == "BLOCKED"
with_fallback = no_safe.model_copy(update={"request_id": "with-fallback", "fallback_candidates": [base]})
repaired = full.evaluate(with_fallback)
if not EXTRA_PACKS:
    assert repaired.results and repaired.results[0].item_id == base.item_id
print("Risk limit, empty feasible set, and checked fallback assertions passed.")

# Isolate policy enforcement from ranking: submit one unsafe canonical plan at a time.
admission = []
for case in cases[:8]:
    row = {"fixture": case["kind"]}
    for label in CONFIGS:
        test_engine = engine_for(label)
        one = Request(tenant_id=TENANT, request_id=f"isolate-{label}-{case['kind']}", purpose=PURPOSE,
                      subject_ref=case["context"], k=1, candidates=[canonical_candidate(case["proposals"][0])])
        verdict = test_engine.evaluate(one)
        row[label] = "REJECT" if not verdict.results else "ADMIT"
        if label == "full": assert not verdict.results
    admission.append(row)
show_table(admission)

# Risk interval: new versioned check with a deliberately tighter threshold (a drop-in YAML file).
narrow = engine_for("full", ("extensions/interval_v2.yaml",))
assert any("WIDE_INTERVAL" in r.reason for r in narrow.evaluate(req).rejections)
# Exact operational boundary: 80% per-line change is allowed; 88% was rejected above.
boundary = replace(sample, lines=tuple(replace(x, amount=22_500 if x.channel == "search" else 2_500 if x.channel == "video" else 12_500) for x in sample.lines))
assert max(abs(x.amount - 12_500) for x in boundary.lines) == 12_500 * RULES.max_line_change
# Compile the same YAML check against an isolated plan table; the engine's trusted data stays unchanged.
boundary_id = "boundary-plan"
boundary = replace(boundary, plan_id=boundary_id, context_id=req.subject_ref)
boundary_check = load_pack(POLICY_DIR / "operations.yaml").compile({**CTX, "PLANS": MappingProxyType({boundary_id: boundary})})["operations-v1"]
assert boundary_check(Candidate(item_id=boundary_id, tenant_id=TENANT, score=0), req) is None

# Prevent replaying one context's approved plan in a different planning request.
cross_context = req.model_copy(update={"request_id": "context-mismatch", "subject_ref": "context-001"})
assert any("CONTEXT_MISMATCH" in r.reason for r in full.evaluate(cross_context).rejections)
spoofed = req.model_copy(update={"request_id": "digest-mismatch", "candidates": [base.model_copy(update={"evidence_ids": ["forged"]})]})
assert any("PLAN_BINDING_MISMATCH" in r.reason for r in full.evaluate(spoofed).rejections)


## 7. Bind the receipt to the delivered allocation

RecGuard signs candidate IDs and attributes, not arbitrary external JSON. Delivering a plan therefore requires a digest check against the approved result, in addition to receipt verification. Changing a spend line after approval must fail. Changes to business rules must invalidate prior receipts before execution; do not place orders merely because a notebook printed a plan.

In [ ]:
def verify_delivery(engine, decision, plan):
    approved = next((c for c in decision.results if c.item_id == plan.plan_id), None)
    return bool(approved and approved.evidence_ids == [digest(plan)]
                and engine.verify(TENANT, decision.decision_id, decision.receipt, decision.results))

if repaired.results:
    delivered = PLANS[repaired.results[0].item_id]
    assert verify_delivery(full, repaired, delivered)
    altered = replace(delivered, lines=(replace(delivered.lines[0], amount=999_999),) + delivered.lines[1:])
    assert not verify_delivery(full, repaired, altered)
    # Loading another pack changes the check fingerprint, even one that admits everything.
    full.load_policy_pack(POLICY_DIR / "extensions/release_review_v2.yaml", context=CTX)
    assert not verify_delivery(full, repaired, delivered)
    print("Original plan verifies; tampered allocation and superseded receipt are rejected.")
assert full.db.audit_verify(TENANT)

## 8. Export and extend

Files are written beside this notebook in `mmm_benchmark_outputs/`. `manifest.json` records rule values, seed, scenario counts, and model/data snapshots. `results.csv` contains request-level measurements; `summary.csv` contains configuration aggregates.

Useful additions: channel-specific minimums, regional growth objectives, inventory-constrained revenue, spend-to-capacity limits, experiment holdouts, brand versus performance envelopes, and approval thresholds. Each new policy needs a reason code, a versioned check in a YAML pack, an independent oracle rule, an unsafe fixture, a feasible boundary fixture, and an abstention case. For cross-plan comparisons or optimization, generate feasible whole plans upstream; custom checks only reject candidates.

For a real MMM, replace `forecast` with backend predictions from a fitted, validated model and actual posterior draws; use real baseline allocations and reporting currency; validate causal assumptions and carryover beyond the decision horizon. Keep the benchmark fixtures and hold out observed periods. An optional Gemini planner should propose typed allocations only: the backend must validate the grid, recompute predictions, and register a frozen snapshot before policy evaluation.

In [ ]:
OUTPUT_DIR = Path("mmm_benchmark_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
for filename, rows in (("results.csv", records), ("summary.csv", summary), ("admission.csv", admission)):
    with (OUTPUT_DIR / filename).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
manifest = {"seed": SEED, "rules": asdict(RULES), "model": MODEL_VERSION, "data": DATA_VERSION,
            "currency": "INR", "cases": len(cases), "plans": len(PLANS), "draws": len(DRAWS),
            "policy_packs": {label: list(packs) for label, packs in CONFIGS.items()}, "extra_packs": list(EXTRA_PACKS),
            "scope": "Synthetic policy benchmark; finite candidate-pool oracle; not causal model validation"}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("Saved benchmark artifacts to", OUTPUT_DIR.resolve())